In [2]:
import pandas as pd
from collections import defaultdict

# =============================
# 1. 参数
# =============================
INPUT_FILE = "./3.cGEP_topgene/ALL_cGEP_top_genes_long_wide_by_cGEP1_70.csv"
OUT_FILE = "./3.cGEP_topgene/cGEP1_70_function_CD8_annotation.csv"

# =============================
# 2. CD8 T marker 基因集
# =============================
marker_dict = {
    "Naive/Memory": {
        "CCR7", "IL7R", "LTB", "TCF7", "LEF1"
    },
    "Cytotoxic": {
        "GZMB", "GNLY", "NKG7", "PRF1", "GZMH"
    },
    "Exhausted": {
        "PDCD1", "HAVCR2", "LAG3", "TOX", "CTLA4"
    },
    "Progenitor_exhausted": {
        "TCF7", "LEF1", "TOX"
    },
    "Cycling": {
        "MKI67", "TOP2A", "HMGB2", "UBE2C", "CENPF"
    },
    "IFN_response": {
        "IFIT1", "IFIT3", "ISG15", "MX1", "STAT1"
    },
    "Trm": {
        "ITGAE", "ZNF683", "CXCR6"
    },
    "Stress": {
        "HSPA1A", "HSP90AA1", "DNAJB1"
    }
}

RIBOSOME_PREFIX = ("RPL", "RPS")

# =============================
# 3. 读取 wide cGEP 表
# =============================
df = pd.read_csv(INPUT_FILE)

# =============================
# 4. 注释函数
# =============================
def annotate_cgep(gene_list):
    genes = set(gene_list)

    # ---- 核糖体程序判断
    ribo_genes = [g for g in genes if g.startswith(RIBOSOME_PREFIX)]
    if len(ribo_genes) / len(genes) > 0.6:
        return {
            "Main_function": "Ribosome / housekeeping",
            "CD8T_state": "Housekeeping",
            "Key_markers": ",".join(ribo_genes[:5])
        }

    # ---- marker 命中统计
    hit_counts = {}
    hit_genes = defaultdict(list)

    for state, markers in marker_dict.items():
        hits = genes & markers
        if hits:
            hit_counts[state] = len(hits)
            hit_genes[state] = list(hits)

    if not hit_counts:
        return {
            "Main_function": "Unknown",
            "CD8T_state": "Unknown",
            "Key_markers": ""
        }

    # ---- 选择命中最多的 CD8 状态
    best_state = max(hit_counts, key=hit_counts.get)

    function_map = {
        "Naive/Memory": "Memory differentiation",
        "Cytotoxic": "Cytotoxicity",
        "Exhausted": "T cell exhaustion",
        "Progenitor_exhausted": "Progenitor exhaustion",
        "Cycling": "Cell cycle",
        "IFN_response": "Interferon response",
        "Trm": "Tissue residency",
        "Stress": "Stress response"
    }

    return {
        "Main_function": function_map.get(best_state, best_state),
        "CD8T_state": best_state,
        "Key_markers": ",".join(hit_genes[best_state][:5])
    }

# =============================
# 5. 对每个 cGEP 做注释
# =============================
results = []

for cgep in df.columns:
    genes = (
        df[cgep]
        .dropna()
        .astype(str)
        .tolist()
    )

    anno = annotate_cgep(genes)

    results.append({
        "cGEP": cgep,
        "Main_function": anno["Main_function"],
        "CD8T_state": anno["CD8T_state"],
        "Key_markers": anno["Key_markers"],
        "Top_genes": ",".join(genes[:10])
    })

# =============================
# 6. 输出结果
# =============================
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_FILE, index=False)

print("✅ 注释完成")
print(f"输出文件: {OUT_FILE}")
print(f"共注释 {out_df.shape[0]} 个 cGEP")

✅ 注释完成
输出文件: ./3.cGEP_topgene/cGEP1_70_function_CD8_annotation.csv
共注释 70 个 cGEP


In [4]:
CD8T_state = out_df[out_df["CD8T_state"] != "Unknown"]
Main_function = out_df[out_df["Main_function"] != "Unknown"]

In [5]:
len(CD8T_state)

33

In [11]:
out_df[~out_df["CD8T_state"].isin(["Unknown", "Housekeeping"])]

,cGEP,Main_function,CD8T_state,Key_markers,Top_genes
1,cGEP2,Interferon response,IFN_response,"ISG15,IFIT3,MX1,IFIT1,STAT1","ISG15,MX1,IFI6,IFIT3,IFIT1,RSAD2,OAS1,IFI44L,I..."
5,cGEP6,T cell exhaustion,Exhausted,"HAVCR2,CTLA4,LAG3","GAPDH,KRT86,GZMB,ENTPD1,TNFRSF9,TIGIT,TNFRSF18..."
6,cGEP7,Stress response,Stress,"DNAJB1,HSP90AA1,HSPA1A","RP11-1033A18.1,DNAJB1,HSP90AA1,HSPA1A,RP11-485..."
7,cGEP8,Cytotoxicity,Cytotoxic,"GNLY,GZMB","GNLY,S100A4,TMSB10,TRBV7-6,IFITM2,RPL39,S100A6..."
9,cGEP10,Memory differentiation,Naive/Memory,"LTB,IL7R","KLRB1,SLC4A10,NCR3,AMICA1,CEBPD,IL7R,LTB,S100A..."
11,cGEP12,Cytotoxicity,Cytotoxic,"GZMH,GZMB,NKG7","TotalSeqC0954,LOC100996919,NKG7,GZMH,TotalseqC..."
16,cGEP17,Cytotoxicity,Cytotoxic,GNLY,"RP11-354E11.2,CRISPLD1,NTM,HDC,GNLY,TYROBP,XCL..."
18,cGEP19,T cell exhaustion,Exhausted,CTLA4,"TNFRSF18,FOXP3,TNFRSF4,IL2RA,LINC01943,BATF,CT..."
19,cGEP20,Cytotoxicity,Cytotoxic,"GZMH,NKG7","TotalSeqC0951,CCL5,CCL4,NKG7,CST7,DUSP2,TRBV7-..."
20,cGEP21,Cell cycle,Cycling,"MKI67,UBE2C,HMGB2,TOP2A,CENPF","UBE2C,C15ORF53,RRM2,TOP2A,STMN1,PCLAF,TUBA1B,T..."


In [6]:
df

,cGEP1,cGEP2,cGEP3,cGEP4,cGEP5,cGEP6,cGEP7,cGEP8,cGEP9,cGEP10,...,cGEP61,cGEP62,cGEP63,cGEP64,cGEP65,cGEP66,cGEP67,cGEP68,cGEP69,cGEP70
0,TotalSeqC0953,ISG15,CYTB,RPL13P12,MT-ND4,GAPDH,RP11-1033A18.1,GNLY,HBB,KLRB1,...,COL3A1,CD7,LINC01297,SFTPC,TPSAB1,MTRNR2L8,LINC01531,HLA-H,IGHGP,AFF3
1,RP11-864N7.2,MX1,ATP6,TotalSeqC0952,MT-CO2,KRT86,DNAJB1,S100A4,HBD,SLC4A10,...,POSTN,KLRC2,STON1,SFTPA2,CNRIP1,RPS4Y1,SEMA3B,CYTB,ID4,ZFP41
2,RP11-234A1.1,IFI6,COX3,EEF1A1,MT-CO3,GZMB,HSP90AA1,TMSB10,AHSP,NCR3,...,COL1A1,KRT81,SLC52A3,SFTPA1,GATA2,DDX3Y,PANK1,ND4,LY6K,MT.ND3
3,RP11-425L10.1,IFIT3,ND4L,TotalSeqC0955,MT-ATP6,ENTPD1,HSPA1A,TRBV7-6,SNCA,AMICA1,...,COL1A2,KRT86,FGFR3,SCGB3A2,RP11-354E11.2,H3F3B,GPR37,HLA-A,TUBB,BEND6
4,RPL13P12,IFIT1,COX2,RPL13,MT-CYB,TNFRSF9,RP11-485G4.2,IFITM2,CA1,CEBPD,...,COL6A1,KLRC3,MSMB,KRT8,PRSS57,MALAT1,ADSSL1,ND4L,RPS20,EVI5
5,RPS12,RSAD2,ND1,RPL34,MT-CO1,TIGIT,HSPA1B,RPL39,HBA2,IL7R,...,COL6A3,LOC101060038,SLC2A5,FTH1,PNMT,CCNL1,RP11-176H8.1,ATP6,RPL29,PLEKHG6
6,TotalseqC0251,OAS1,ND5,TPT1,MT-ND2,TNFRSF18,HSPH1,S100A6,ALAS2,LTB,...,CAPN6,ZNF683,PPARGC1A,AL162511.1,HPGDS,NR4A2,EMILIN1,ATP8,RAB13,UBD
7,RPL13,IFI44L,ND2,RPL32,MT-ND1,CD7,HSP90AB1,TRAV8-1,HBA1,S100A4,...,LUM,CD9,AGR2,KRT18,MS4A2,SRSF7,LINC00839,ND2,TM4SF1,MT.ND2
8,RPL32,ISG20,ATP8,RPS12,MT-ND3,LAYN,HSPE1,ATP5E,SLC4A1,DUSP1,...,SGCD,ND4L,VANGL2,LINC01986,CPA3,CLK1,S100A1,ND1,EEF1A1,C2orf88
9,AC024293.1,LY6E,RPS14P3,AC010343.1,MT-ND5,AC092580.4,HSPD1,SH3BGRL3,SELENBP1,CCR6,...,DCN,CTSW,PDLIM3,RPL7,TPSB2,EIF1,ZNF202,NKG7,HES1,BHLHB9
